# 01 — LLM Behavior and Prompt Anatomy

## Scenario
Northstar’s classifier changes behavior after a request configuration change. This lab treats the request packet as an observable system and tests one variable at a time.

**Safety boundary:** this is an experimental simulator using the `google-genai` SDK. Its effects illustrate an experimental method of treating the prompt as a configurable packet.

In [ ]:
import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()

MODEL_ID = 'gemini-2.5-flash'

class SupportClassification(BaseModel):
    category: str = Field(description="One of: refund, shipping, account, unknown")

CASES = [
    {'id': 'clear-refund', 'message': 'I want to return my order #123.', 'expected': 'refund'},
    {'id': 'clear-shipping', 'message': 'Where is my package?', 'expected': 'shipping'},
    {'id': 'clear-account', 'message': 'I need to reset my password.', 'expected': 'account'},
    {'id': 'ambiguous-payment', 'message': 'Why did you charge me twice?', 'expected': 'unknown'}
]

EVIDENCE = """Approved Policies:\n- Refunds: Allowed within 30 days of purchase.\n- Shipping: Track via the carrier link in your email.\n- Account: Reset passwords via the login page.\n"""

## Baseline hypothesis

A precise classification instruction, approved evidence, and low variation (temperature 0) should classify clear cases while escalating the ambiguous payment case. We will score that baseline before changing anything.

In [ ]:
def run_classification_suite(instruction, evidence, position='first', temperature=0.0):
    results = []
    total_tokens = 0
    
    for case in CASES:
        if position == 'first':
            prompt = f"{instruction}\n\nEvidence:\n{evidence}\n\nMessage: {case['message']}"
        else: # middle
            synthetic_padding = "The customer is a highly valued member. Please be polite.\n" * 10
            prompt = f"{instruction}\n\n{synthetic_padding}Evidence:\n{evidence}\n{synthetic_padding}\nMessage: {case['message']}"
            
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temperature,
                response_mime_type="application/json",
                response_schema=SupportClassification,
            )
        )
        
        output = response.text
        import json
        try:
            category = json.loads(output).get('category', 'unknown')
        except:
            category = 'unknown'
            
        results.append({
            'id': case['id'],
            'expected': case['expected'],
            'observed': category
        })
        if response.usage_metadata:
            total_tokens += response.usage_metadata.total_token_count
            
    accuracy = sum(1 for r in results if r['expected'] == r['observed']) / len(results)
    return results, {'accuracy': accuracy, 'mean_tokens': total_tokens / len(CASES)}

instruction = "Classify the support request using ONLY the approved evidence. If the message doesn't match the evidence clearly, return 'unknown'."
baseline_results, baseline_metrics = run_classification_suite(instruction, EVIDENCE, position='first', temperature=0.0)
print("Baseline Results:", baseline_results)
print("Baseline Metrics:", baseline_metrics)

## Experiment 1 — position is a variable

Keep the cases and instruction fixed. Move the evidence into the synthetic middle position. This tells us to test source order on a real model, not to assume a universal effect.

In [ ]:
middle_results, middle_metrics = run_classification_suite(instruction, EVIDENCE, position='middle', temperature=0.0)
print("Middle Position Results:", middle_results)
print("Middle Position Metrics:", middle_metrics)

## Experiment 2 — sampling is a trade-off

Now keep evidence first but introduce a non-zero temperature. On a real model, run repeated samples and compare variation, task accuracy, and the cost of additional calls. Never infer correctness from lower temperature alone.

In [ ]:
varied_results, varied_metrics = run_classification_suite(instruction, EVIDENCE, position='first', temperature=1.0)
print("High Temp Results:", varied_results)
print("High Temp Metrics:", varied_metrics)

print("\n--- Comparison ---")
print(f"Baseline Accuracy: {baseline_metrics['accuracy']} (Tokens: {baseline_metrics['mean_tokens']})")
print(f"Middle Pos Accuracy: {middle_metrics['accuracy']} (Tokens: {middle_metrics['mean_tokens']})")
print(f"High Temp Accuracy: {varied_metrics['accuracy']} (Tokens: {varied_metrics['mean_tokens']})")

## Failure injection — missing evidence

A refund decision without approved evidence must not become a confident refund label. This is a context/contract failure, not a request for stronger role wording. The safe result is `unknown`, followed by clarification or escalation in the surrounding application.

In [ ]:
# DELIBERATE FAILURE: Passing NO evidence but expecting the model to abstain.
instruction_no_fallback = "Classify the support request."

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"{instruction_no_fallback}\n\nMessage: Can I return my order?",
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=SupportClassification,
    )
)

import json
output = json.loads(response.text)
print(f"Without evidence, the model hallucinates a confident category: {output['category']}")

assert output['category'] != 'unknown', "The model successfully abstained, which breaks the failure mode demonstration."

## The Fix: Strong Instruction and Abstention

We must instruct the model to abstain if the evidence does not support a clear category.

In [ ]:
instruction_with_fallback = "Classify the support request using ONLY the provided evidence. If no evidence is provided or it doesn't clearly match, return 'unknown'."

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"{instruction_with_fallback}\n\nEvidence: None\n\nMessage: Can I return my order?",
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=SupportClassification,
    )
)

import json
output = json.loads(response.text)
print(f"With a strong abstention instruction, the model safely returns: {output['category']}")

assert output['category'] == 'unknown', "The model failed to abstain."
print("\nSuccess! The failure was fixed.")